In [0]:
# Silver layer: companies
# Source: b2b_ecommerce.bronze.companies (append-only bronze ingestion)
# Grain:  one row per company_id and version (history via __START_AT/__END_AT)
# Approach: bronze.companies is append-only, so we stream it, clean +
#           standardize each row, and merge into b2b_ecommerce.silver.companies
#           with AUTO CDC (merge on company_id, sequenced by updated_at,
#           SCD Type 2 = keep the full change history).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_COMPANIES = "b2b_ecommerce.bronze.companies"

CUIT_FORMAT = "^[0-9]{2}-[0-9]{8}-[0-9]$"  # XX-XXXXXXXX-X


def clean_text(name, lower=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if lower:
        trimmed = F.lower(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned + standardized stream of bronze.companies
# ---------------------------------------------------------------------------
@dp.temporary_view(name="companies_clean_stream")
# Data quality validation (only checks that can actually catch bad rows)
@dp.expect_or_drop("company_id_not_null", "company_id IS NOT NULL")  # merge key must exist
@dp.expect_or_drop("updated_at_not_null", "updated_at IS NOT NULL")   # CDC sequence must exist
@dp.expect("name_not_null", "name IS NOT NULL")
@dp.expect("company_type_valid", "company_type IN ('buyer', 'supplier')")
@dp.expect("contact_email_valid", "contact_email IS NOT NULL AND contact_email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'")
@dp.expect("phone_format_valid", "phone IS NULL OR phone RLIKE '^[+][0-9]'")  # international format
@dp.expect("rating_range_valid", "rating IS NULL OR rating BETWEEN 0 AND 5")
@dp.expect("employee_count_not_negative", "employee_count IS NOT NULL AND employee_count >= 0")
def companies_clean_stream():
    # bronze.companies is append-only: every change arrives as a new row.
    # The AUTO CDC merge below turns each change into a new version in the
    # SCD Type 2 history (sequenced by updated_at).
    bronze = spark.readStream.table(BRONZE_COMPANIES)

    cuit = F.trim(F.col("cuit"))

    return bronze.select(
        # Column types: explicit casts keep the silver schema stable
        F.col("company_id").cast("int").alias("company_id"),
        # Cleaning: trim whitespace; blank text -> NULL
        clean_text("name"),
        # Invalid-value handling: keep only well-formed CUITs (XX-XXXXXXXX-X);
        # placeholders ('N/A', all-zeros) and junk formats -> NULL
        F.when(cuit.rlike(CUIT_FORMAT) & (cuit != "00-00000000-0"), cuit)
        .otherwise(F.lit(None).cast("string"))
        .alias("cuit"),
        # Standardization: lowercase categories for consistent grouping
        clean_text("company_type", lower=True),
        clean_text("country", lower=True),
        clean_text("city"),
        F.col("rating").cast("decimal(2,1)").alias("rating"),
        clean_text("address"),
        clean_text("phone"),  # valid intl. numbers (+54/+55/+598...)
        clean_text("contact_email", lower=True),
        clean_text("industry", lower=True),
        F.col("employee_count").cast("int").alias("employee_count"),
        clean_text("payment_terms", lower=True),
        F.col("is_active").cast("boolean").alias("is_active"),
        F.col("created_at").cast("timestamp").alias("created_at"),
        # updated_at drives the CDC merge ordering; fall back to created_at
        # if a bronze row ever arrives without one.
        F.coalesce(F.col("updated_at"), F.col("created_at"))
        .cast("timestamp")
        .alias("updated_at"),
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.companies
#    Rows are merged by company_id; when the same company is appended again
#    with a newer updated_at, a new version is kept (full history).
#    Current state per company = the row where __END_AT IS NULL.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="companies",
    comment=(
        "Silver companies: cleaned, standardized; full history per company_id "
        "with __START_AT/__END_AT validity (SCD Type 2)"
    ),
)

dp.create_auto_cdc_flow(
    target="companies",
    source="companies_clean_stream",
    keys=["company_id"],
    sequence_by="updated_at",
    stored_as_scd_type=2,
)
